# CeNN Cellular Attention — Sparse / Shifted Windows vs Transformer

**Goal:** test whether a causal 1-D Cellular Attention layer can remove the local receptive-field bottleneck while staying much sparser than full Transformer attention.

This notebook follows the same discipline as `CeNN_Research_Layers_Colab.ipynb`: frozen SmolLM2-135M, document-disjoint train/validation/test splits, attention-output transfer, next-token refinement, **validation-only candidate selection**, and held-out paired NLL intervals.

### The receptive-field bottleneck

A pure local CeNN with a small causal kernel can move information only a few tokens per recurrent step. Dense Transformer attention has a one-layer path to every earlier token, but creates a quadratic number of score pairs.

**Cellular Attention changes the neighborhood across cellular steps.** With power-of-two dilations,

`1, 2, 4, 8, 16, ...`

each step reaches a new distance scale. A causal 3-cell neighborhood uses positions

`i, i-d, i-2d`

and never reads future tokens. With eight steps through dilation 128, the theoretical receptive field is **511 tokens**.

| Candidate | Neighborhood | Why test it |
|---|---|---|
| `cellular_local3` | `{0,1,2}` every step | control: traditional local bottleneck |
| `cellular_dilated3` | `{0,d,2d}` | minimal exponential-distance attention |
| `cellular_dilated5` | `{0,d,2d,3d,4d}` | more capacity at each scale |
| `cellular_multiscale5` | `{0,1,d,2d,4d}` | local syntax + long-range sparse spokes |
| `cellular_shifted8` | alternating causal 8-token shifted blocks | shifted-window alternative |

For an extended schedule with `S = O(log N)` steps and fixed small neighborhood width `w`, the score computation is `O(N · w · log N)` rather than dense `O(N²)`. The PyTorch reference is not expected to beat fused SDPA on raw speed yet; the experiment measures **quality, sparsity, peak memory and actual timing separately**.


## 1 · Fresh checkout and dependencies

In [ ]:
import importlib, pathlib, subprocess, sys, tempfile

REPO_REF = "main"
WORK_PARENT = pathlib.Path("/content") if pathlib.Path("/content").exists() else pathlib.Path.cwd()
REPO_DIR = pathlib.Path(tempfile.mkdtemp(prefix="TinyCeNN-cellular-", dir=WORK_PARENT))

subprocess.run([
    "git", "clone", "--depth", "1", "--branch", REPO_REF,
    "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO_DIR)
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-e", str(REPO_DIR),
    "transformers==4.57.6", "datasets>=3,<5",
    "pandas", "matplotlib", "pytest>=8"
], check=True)

for path in (REPO_DIR, REPO_DIR / "src"):
    sys.path.insert(0, str(path))
importlib.invalidate_caches()

SOURCE_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
print("Source commit:", SOURCE_COMMIT)
print("Checkout:", REPO_DIR)


## 2 · Choose the experiment budget

`balanced` is the recommended first run. It evaluates **five architectural scenarios × two feature sizes** at layer 18. `smoke` only checks the pipeline; `extended` increases context, data and training budget.

The power-of-two dilation schedule is chosen so the recurrent sparse graph can cover the longer test context through multiple cellular hops.


In [ ]:
import torch
from datetime import datetime, timezone

PROFILE = "balanced"  # smoke | balanced | extended
LAYERS = "18"
VARIANTS = (
    "cellular_local3,cellular_dilated3,cellular_dilated5,"
    "cellular_multiscale5,cellular_shifted8"
)
SEED = 2026
ALLOW_CPU = False
SAVE_TO_DRIVE = False

PROFILES = {
    "smoke": dict(
        context=64, test_contexts="64,128", feature_dims="24",
        dilations="1,2,4,8,16,32",
        train_documents=8, validation_documents=4, test_documents=4,
        steps=12, lm_steps=2, eval_every=6, shifted_window=8,
    ),
    "balanced": dict(
        context=256, test_contexts="256,512", feature_dims="32,64",
        dilations="1,2,4,8,16,32,64,128",
        train_documents=64, validation_documents=12, test_documents=24,
        steps=300, lm_steps=40, eval_every=50, shifted_window=8,
    ),
    "extended": dict(
        context=512, test_contexts="512,1024", feature_dims="64,96",
        dilations="1,2,4,8,16,32,64,128,256",
        train_documents=192, validation_documents=32, test_documents=64,
        steps=900, lm_steps=100, eval_every=100, shifted_window=16,
    ),
}
CONFIG = PROFILES[PROFILE].copy()

assert torch.cuda.is_available() or ALLOW_CPU, (
    "Select Runtime → Change runtime type → GPU, or use ALLOW_CPU=True for smoke only."
)
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Profile:", PROFILE, CONFIG)
print("Candidate runs:",
      len(LAYERS.split(",")) * len(VARIANTS.split(",")) * len(CONFIG["feature_dims"].split(",")))

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULT_ROOT = pathlib.Path("/content/drive/MyDrive/TinyCeNN-LM/cellular-attention")
else:
    RESULT_ROOT = WORK_PARENT / "TinyCeNN-cellular-results"
RESULT_ROOT.mkdir(parents=True, exist_ok=True)


## 3 · Validate the new layer before using pretrained weights

These tests check output shape, finite gradients, strict causality, exponential receptive-field growth, sparse score-pair counts and checkpoint reconstruction.


In [ ]:
subprocess.run([
    sys.executable, "-m", "pytest", "-q",
    str(REPO_DIR / "tests/test_cellular_attention.py"),
    str(REPO_DIR / "tests/test_research_layer_benchmark.py"),
], cwd=REPO_DIR, check=True)


## 4 · Train and evaluate the Cellular Attention candidates

The pretrained model is frozen. Only the new Cellular Attention layer is optimized.

1. Match the original attention output.
2. Refine the replacement using next-token cross entropy.
3. Select the candidate with the lowest validation NLL.
4. Freeze that decision.
5. Evaluate every preregistered candidate on untouched test documents.

The exact-softmax replacement control must agree with the original model before candidate training begins.


In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT_DIR = RESULT_ROOT / f"{PROFILE}-{RUN_ID}"

command = [
    sys.executable, str(REPO_DIR / "scripts" / "benchmark_cellular_attention.py"),
    "--layers", LAYERS,
    "--variants", VARIANTS,
    "--seed", str(SEED),
    "--output-dir", str(OUTPUT_DIR),
]
for key, value in CONFIG.items():
    command.extend(["--" + key.replace("_", "-"), str(value)])

print("Results:", OUTPUT_DIR)
print("Command:", subprocess.list2cmdline(command))
subprocess.run(command, cwd=REPO_DIR, check=True)


## 5 · Decision table

The strongest result is `strict_quality_win=True`: the entire paired 95% bootstrap interval for candidate-minus-Transformer NLL is below zero.

`score_pair_ratio` compares the number of sparse Cellular Attention score pairs with dense causal Transformer score pairs. A value of `0.10` means only 10% as many score pairs inside the attention core.


In [ ]:
import json, pandas as pd
from IPython.display import display

report = json.loads((OUTPUT_DIR / "cellular_attention_report.json").read_text())
results = pd.read_csv(OUTPUT_DIR / "cellular_attention_summary.csv")
validation = pd.read_csv(OUTPUT_DIR / "validation_summary.csv")
history = pd.read_csv(OUTPUT_DIR / "training_history.csv")

candidates = results[results["variant"] != "transformer_original"].copy()
selected = candidates[candidates["selected_on_validation"].eq(True)]

print("Validation-selected candidates:", report["validation_winners"])
display(selected[[
    "candidate", "context", "test_perplexity", "transformer_perplexity",
    "ppl_ratio", "delta_nll", "delta_nll_ci_low", "delta_nll_ci_high",
    "score_pair_ratio", "receptive_field_tokens", "strict_quality_win", "quality"
]].round(6))

print("All preregistered candidates:")
display(candidates[[
    "variant", "feature_dim", "context", "selected_on_validation",
    "trainable_parameters", "test_perplexity", "ppl_ratio",
    "output_cosine", "grad_mean_cosine", "score_pair_ratio",
    "prefill_speedup", "peak_extra_bytes", "quality"
]].round(6))


## 6 · Visualize quality versus sparsity

In [ ]:
import matplotlib.pyplot as plt

view = candidates[candidates["context"].eq(CONFIG["context"])].copy()

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(view["score_pair_ratio"], view["delta_nll"], s=90)
for _, row in view.iterrows():
    label = f"{row['variant']} f{int(row['feature_dim'])}"
    ax.annotate(label, (row["score_pair_ratio"], row["delta_nll"]),
                xytext=(5, 5), textcoords="offset points", fontsize=8)
ax.axhline(0, linewidth=1)
ax.axhline(0.02, linewidth=1, linestyle="--")
ax.axhline(-0.02, linewidth=1, linestyle="--")
ax.set_xlabel("Sparse score pairs / dense causal Transformer score pairs")
ax.set_ylabel("ΔNLL vs Transformer (lower is better)")
ax.set_title("Cellular Attention: quality–sparsity frontier")
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
ordered = view.sort_values("ppl_ratio")
labels = [f"{v}\nf{int(f)}" for v, f in zip(ordered["variant"], ordered["feature_dim"])]
ax.bar(labels, ordered["ppl_ratio"])
ax.axhline(1.0, linewidth=1)
ax.set_ylabel("Perplexity ratio vs Transformer")
ax.set_title("Replacement quality at training context")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()


## 7 · How to interpret a win

A single-layer held-out win is evidence that the sparse cellular graph can preserve or improve the next-token objective **for this replacement experiment**. It is not yet evidence that a full Cellular Attention language model beats Transformers.

If `cellular_multiscale5` or `cellular_dilated5` wins, the next rigorous experiment should replace multiple layers and add a matched trainable-Transformer control with the same adaptation budget.


## 8 · Download the complete results

This final cell packages **everything** from the run—reports, CSVs, checkpoints, token blocks, selection file and manifest—into one ZIP and automatically starts the browser download in Colab.


In [ ]:
import shutil
from pathlib import Path

ZIP_BASE = WORK_PARENT / f"cellular-attention-{PROFILE}-{RUN_ID}-results"
ZIP_PATH = Path(shutil.make_archive(
    str(ZIP_BASE), "zip", root_dir=str(OUTPUT_DIR)
))
print("Created:", ZIP_PATH)
print("ZIP size (MB):", round(ZIP_PATH.stat().st_size / (1024**2), 2))

try:
    from google.colab import files
    files.download(str(ZIP_PATH))
except Exception as exc:
    print("Automatic Colab download unavailable:", exc)
    print("Result ZIP:", ZIP_PATH)
